# SpendDNA — Your Wallet's Year-End Story
### *"Spotify Wrapped for your money."*

**Name:** SACHIN R B

**Leaner_ID:** 17257

**Batch:** Unlox Academy — Data Science Program

**Date:** Week 2 — Industry-Graded Minor Project
**Dataset:**
`rahul_transactions.csv` as named as `Data set for DADS June.csv` (6 months, Jan–Jun 2024, Bengaluru software engineer Rahul Sharma)

##Given
**Constraints followed:** No pandas-profiling/sweetviz, no matplotlib/seaborn/plotly, no scikit-learn/scipy.stats,
no `collections.Counter`, no `re` (regex). Only Python fundamentals, NumPy, and Pandas (`.str`, `.dt`, `groupby`,
`pivot_table`, `transform`) as permitted by the brief.

## Feature 1 — The Transaction Parser
Reading the raw CSV, handles 4 date formats, 3 amount formats, standardises `Type`, and drops exact duplicates.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv('Data set for DADS June.csv') # rahul transaction csv
print(f"Raw rows: {len(raw)}")

df = raw.copy()

# standardise type
df['type_clean'] = df['Type'].str.lower().replace({'dr': 'debit', 'cr': 'credit'})

# clean amount
df['amount_clean'] = (
    df['Amount'].astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['amount_clean'] = pd.to_numeric(df['amount_clean'], errors='coerce')

# parse date — try each known format explicitly (avoids dayfirst/dateutil ambiguity)
KNOWN_DATE_FORMATS = ['%d/%m/%y', '%Y-%m-%d', '%d-%b-%y', '%d %b %Y']
date_clean = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
for fmt in KNOWN_DATE_FORMATS:
    still_unparsed = date_clean.isna()
    date_clean.loc[still_unparsed] = pd.to_datetime(df.loc[still_unparsed, 'Date'], format=fmt, errors='coerce')
df['date_clean'] = date_clean

df['Mode'] = df['Mode'].replace('', np.nan)

# drop duplicates
rows_before = len(df)
df = df.drop_duplicates(subset=['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref'])
duplicates_dropped = rows_before - len(df)

unparseable_amount = int(df['amount_clean'].isna().sum())
unparseable_date = int(df['date_clean'].isna().sum())
df = df.dropna(subset=['amount_clean', 'date_clean']).reset_index(drop=True)

df_clean = df[['date_clean', 'Time', 'Description', 'type_clean', 'amount_clean', 'Balance', 'Mode', 'Ref']].copy()
df_clean.columns = ['date', 'time', 'description', 'type', 'amount', 'balance', 'mode', 'ref']

months_covered = df_clean['date'].dt.to_period('M').nunique()
print(f"Parsed {len(df_clean)} transactions across {months_covered} months.")
print(f"Dropped {duplicates_dropped} duplicates.")
print(f"{unparseable_amount} unparseable amounts, {unparseable_date} unparseable dates.")
print(df_clean.dtypes)
df_clean.head()

Raw rows: 1328
Parsed 1310 transactions across 6 months.
Dropped 18 duplicates.
0 unparseable amounts, 0 unparseable dates.
date           datetime64[ns]
time                   object
description            object
type                   object
amount                float64
balance               float64
mode                   object
ref                    object
dtype: object


,date,time,description,type,amount,balance,mode,ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,debit,2462.0,678275.0,UPI,TXN190872
1,2024-01-01,05:44,BHIM-BMTC,debit,50.0,681007.0,UPI,TXN143064
2,2024-01-01,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84728.0,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,debit,1828.0,-748745.0,UPI,TXN569226
4,2024-01-01,14:23,BHIM-BLINKIT,debit,270.0,680737.0,UPI,TXN968962


## Feature 2 — Vendor Extractor

This one took me the longest. The bank doesn't just write "Swiggy" — it's stuff like UPI-SWIGGY-1234@HDFCBANK, or even weirder, BUNDL Tech P L which turns out is Swiggy's actual parent company name. So I built a dictionary of keywords for every vendor and made sure to check the more specific ones first — like checking for "AMAZON PRIME" before just "AMAZON" — otherwise everything Amazon-related would've gotten lumped together wrong.

In [2]:
print(f"Unique descriptions: {df_clean['description'].nunique()}")
print(df_clean['description'].unique()[:20])

Unique descriptions: 283
['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT']


In [3]:
# specific patterns before generic parents (e.g. AMAZON PRIME before AMAZON)
VENDOR_KEYWORDS = {
    'Amazon Prime':        ['AMAZONPRIME', 'AMAZON PRIME'],
    'Swiggy Instamart':    ['SWIGGYINSTAMART', 'SWIGGY INSTAMART'],
    'Swiggy':              ['SWIGGY', 'BUNDL'],
    'Zomato':              ['ZOMATO'],
    'Zepto':               ['ZEPTO'],
    'Blinkit':             ['BLINKIT', 'GROFERS'],
    'Amazon':              ['AMAZON', 'AMZN'],
    'Flipkart':            ['FLIPKART', 'FKRT'],
    'Myntra':              ['MYNTRA'],
    'Ajio':                ['AJIO', 'RELIANCE RETAIL'],
    'Uber':                ['UBER'],
    'Ola':                 ['OLACABS', 'OLA CABS', 'ANI TECHNOLOGIES'],
    'Rapido':              ['RAPIDO', 'ROPPEN'],
    'Starbucks':           ['STARBUCKS'],
    'Third Wave Coffee':   ['THIRDWAVECOFFEE', 'THIRD WAVE COFFEE'],
    'Blue Tokai':          ['BLUETOKAI', 'BLUE TOKAI'],
    'Truffles':            ['TRUFFLES'],
    'Empire Restaurant':   ['EMPIRE RESTAURANT', 'EMPIRERESTAURANT'],
    'Meghana Foods':       ['MEGHANA'],
    'Toit Brewpub':        ['TOIT'],
    'Netflix':             ['NETFLIX'],
    'Spotify':             ['SPOTIFY'],
    'Hotstar':             ['HOTSTAR', 'NOVI DIGITAL'],
    'YouTube Premium':     ['YOUTUBEPREMIUM', 'YOUTUBE PREMIUM'],
    'BigBasket':           ['BIGBASKET', 'SUPERMARKET GROCERY'],
    'DMart':               ['DMART', 'AVENUE SUPERMARTS'],
    'More Supermarket':    ['MORE SUPERMARKET', 'MORERETAIL'],
    'Zerodha':             ['ZERODHA'],
    'Groww':               ['GROWW', 'NEXTBILLION'],
    'Indian Oil':          ['INDIAN OIL', 'IOCL'],
    'HP Petrol':           ['HP PETROL', 'HINDUSTAN PETROLEUM'],
    'Shell':               ['SHELL'],
    'BookMyShow':          ['BOOKMYSHOW', 'BIGTREE'],
    'PVR Cinemas':         ['PVR'],
    'Salary':              ['SALARY'],
    'Cash Withdrawal':     ['ATM-WDL', 'ATM'],
    'BESCOM':              ['BESCOM'],
    'Airtel':              ['AIRTEL'],
    'Jio':                 ['JIO'],
    'BWSSB Water':         ['BWSSB'],
    'P2P Transfer':        ['LANDLORD', 'PRIYA', 'ANKIT', 'ROHAN', 'SNEHA', 'VIKRAM',
                             'DEEPA', 'KARAN', 'NEHA', 'ARJUN', 'POOJA'],
}

def extract_vendor(description, keyword_map):
    desc_upper = str(description).upper()
    for vendor, keywords in keyword_map.items():
        for kw in keywords:
            if kw in desc_upper:
                return vendor
    return 'Uncategorised'

df_clean['vendor_clean'] = df_clean['description'].apply(lambda d: extract_vendor(d, VENDOR_KEYWORDS))

print(f"Canonical vendors found: {df_clean['vendor_clean'].nunique()}")
print(df_clean['vendor_clean'].value_counts().head(10))

Canonical vendors found: 34
vendor_clean
Uncategorised    262
Swiggy           223
Zomato           121
Amazon            82
Uber              71
Zepto             58
Rapido            55
Blinkit           55
Ola               49
Flipkart          43
Name: count, dtype: int64


## Feature 3 — Category Tagger

Once I had clean vendor names, I mapped each one to a category — Food Delivery, Groceries, Investments, whatever it was. This is really what makes the data useful, because now I'm not just looking at "spent ₹450 at Swiggy," I'm looking at "spent this much on Food Delivery overall," which is what actually tells a story.

In [4]:
VENDOR_CATEGORY = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce', 'Swiggy Instamart': 'Quick Commerce',
    'Amazon': 'E-commerce', 'Flipkart': 'E-commerce', 'Myntra': 'E-commerce', 'Ajio': 'E-commerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport',
    'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe', 'Blue Tokai': 'Cafe',
    'Truffles': 'Restaurants', 'Empire Restaurant': 'Restaurants',
    'Meghana Foods': 'Restaurants', 'Toit Brewpub': 'Restaurants',
    'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Amazon Prime': 'Subscriptions',
    'Hotstar': 'Subscriptions', 'YouTube Premium': 'Subscriptions',
    'BESCOM': 'Utilities', 'Airtel': 'Utilities', 'Jio': 'Utilities', 'BWSSB Water': 'Utilities',
    'BigBasket': 'Groceries', 'DMart': 'Groceries', 'More Supermarket': 'Groceries',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'Indian Oil': 'Fuel', 'HP Petrol': 'Fuel', 'Shell': 'Fuel',
    'BookMyShow': 'Entertainment', 'PVR Cinemas': 'Entertainment',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal',
    'Salary': 'Income',
}

df_clean['category'] = df_clean['vendor_clean'].map(VENDOR_CATEGORY)
df_clean['category'] = df_clean['category'].fillna('Uncategorised')

print(f"Categories used: {df_clean['category'].nunique()}")
print(df_clean['category'].value_counts())

Categories used: 16
category
Food Delivery        344
Uncategorised        262
Transport            175
E-commerce           145
Quick Commerce       113
Cafe                  53
Restaurants           34
Groceries             33
Subscriptions         32
Utilities             29
Investments           23
Fuel                  20
Personal Transfer     18
Cash Withdrawal       17
Income                 6
Entertainment          6
Name: count, dtype: int64


## Feature 4 — Spending Overview

This is basically the headline numbers — how much came in, how much went out, and whether I'm actually saving anything. I left out P2P transfers and ATM withdrawals when calculating category percentages since that's not really spending, it's just money moving from one place to another.

In [5]:
credits = df_clean[df_clean['type'] == 'credit']
debits = df_clean[df_clean['type'] == 'debit']

total_credit = credits['amount'].sum()
total_debit = debits['amount'].sum()
net_change = total_credit - total_debit
savings_rate = net_change / total_credit * 100

# exclude P2P/Cash Withdrawal from category % (not "consumption")
debit_spend = debits[~debits['category'].isin(['Personal Transfer', 'Cash Withdrawal'])]

cat_totals = debit_spend.groupby('category')['amount'].sum().sort_values(ascending=False)
cat_pct = cat_totals / cat_totals.sum() * 100

vendor_totals = debit_spend.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False)
vendor_counts = debit_spend.groupby('vendor_clean')['amount'].count()

print("EXECUTIVE SUMMARY")
print(f"  Total credits : Rs. {total_credit:,.0f}")
print(f"  Total debits  : Rs. {total_debit:,.0f}")
print(f"  Net change    : Rs. {net_change:,.0f}")
print(f"  Savings rate  : {savings_rate:.1f}%")
print(f"  Transactions  : {len(df_clean)}")
print(f"  Unique vendors: {df_clean['vendor_clean'].nunique()}")

print("\nTOP 5 CATEGORIES")
for cat, amt in cat_totals.head(5).items():
    print(f"  {cat:<16} {cat_pct[cat]:5.1f}%   Rs. {amt:,.0f}")

print("\nTOP 5 VENDORS")
for v, amt in vendor_totals.head(5).items():
    print(f"  {v:<16} Rs. {amt:>10,.0f}  ({vendor_counts[v]} txns)")

EXECUTIVE SUMMARY
  Total credits : Rs. 509,774
  Total debits  : Rs. 1,678,901
  Net change    : Rs. -1,169,127
  Savings rate  : -229.3%
  Transactions  : 1310
  Unique vendors: 34

TOP 5 CATEGORIES
  E-commerce        37.4%   Rs. 564,577
  Investments       16.4%   Rs. 248,160
  Uncategorised     13.9%   Rs. 210,491
  Food Delivery     10.0%   Rs. 150,839
  Quick Commerce     4.0%   Rs. 60,152

TOP 5 VENDORS
  Amazon           Rs.    324,303  (82 txns)
  Uncategorised    Rs.    210,491  (262 txns)
  Zerodha          Rs.    210,000  (14 txns)
  Flipkart         Rs.    170,745  (43 txns)
  Swiggy           Rs.     95,523  (223 txns)


## Feature 5 — Monthly Trend Analysis

Instead of just looking at totals, I wanted to see how things changed month to month. Like, is Food Delivery spend going up over time or staying flat? That's a more interesting question than just "how much did I spend total."

In [6]:
debit_spend = debit_spend.copy()
debit_spend['month'] = debit_spend['date'].dt.month

month_pivot = debit_spend.pivot_table(values='amount', index='category', columns='month', aggfunc='sum', fill_value=0)
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
month_pivot = month_pivot.rename(columns=month_names)

first_col, last_col = month_pivot.columns[0], month_pivot.columns[-1]
first_vals = month_pivot[first_col].replace(0, np.nan)
growth_pct = ((month_pivot[last_col] - month_pivot[first_col]) / first_vals * 100).dropna()

biggest_growth_cat = growth_pct.idxmax()
biggest_decline_cat = growth_pct.idxmin()

print("MONTHLY SPEND BY CATEGORY (Rs.)")
print(month_pivot.round(0))
print(f"\nTrending UP fastest:   {biggest_growth_cat} ({growth_pct[biggest_growth_cat]:+.1f}%)")
print(f"Trending DOWN fastest: {biggest_decline_cat} ({growth_pct[biggest_decline_cat]:+.1f}%)")

MONTHLY SPEND BY CATEGORY (Rs.)
month               Jan      Feb       Mar      Apr      May       Jun
category                                                              
Cafe             1029.0   2736.0    3056.0   4034.0   3994.0    3745.0
E-commerce      92182.0  83959.0  102937.0  58462.0  91942.0  135095.0
Entertainment     952.0    474.0     562.0   2224.0      0.0       0.0
Food Delivery   22633.0  23740.0   24803.0  27756.0  25408.0   26499.0
Fuel            15351.0   2079.0   13669.0  18013.0   7188.0    2882.0
Groceries       17649.0   7517.0    5523.0   6225.0   7862.0    5432.0
Investments     38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce   9995.0  12459.0   12229.0   8221.0  10139.0    7109.0
Restaurants      6739.0   4744.0   20682.0   3203.0  14368.0    3198.0
Subscriptions    2767.0   4594.0    3897.0   2147.0   1844.0    4597.0
Transport        7951.0   8077.0    5094.0   7799.0  10208.0    5582.0
Uncategorised   47783.0  34537.0   35800.0  2

## Feature 6 — Time-of-Day Patterns

This is about when the spending happens, not just what it's on. I pulled the hour out of every transaction and it clearly showed Food Delivery clustering late at night, while Cafe spending was all in the morning — which honestly matches how most of us actually live.

In [7]:
debit_spend['hour'] = debit_spend['time'].str[:2].astype(int)

hour_matrix = pd.pivot_table(debit_spend, values='amount', index='category', columns='hour', aggfunc='count', fill_value=0)

food = debit_spend[debit_spend['category'] == 'Food Delivery']
food_by_hour = food.groupby('hour').size().reindex(range(24), fill_value=0)
max_count = food_by_hour.max()

print("FOOD DELIVERY ORDERS BY HOUR")
for h, count in food_by_hour.items():
    bar = '#' * int((count / max_count) * 40) if max_count else ''
    print(f"  {h:02d}:00  {bar} {count}")

late_night_pct = food['hour'].isin([21, 22, 23, 0, 1]).mean() * 100
print(f"\n{late_night_pct:.1f}% of Food Delivery orders happen 21:00-02:00.")

cafe = debit_spend[debit_spend['category'] == 'Cafe']
cafe_morning_pct = cafe['hour'].between(8, 11).mean() * 100
print(f"{cafe_morning_pct:.1f}% of Cafe spend happens 08:00-11:00.")

FOOD DELIVERY ORDERS BY HOUR
  00:00  # 2
  01:00  ####### 8
  02:00  # 2
  03:00  ### 4
  04:00  ###### 7
  05:00  ###### 7
  06:00   1
  07:00  ## 3
  08:00  ######## 9
  09:00  ############ 13
  10:00  ###### 7
  11:00  #################### 21
  12:00  ######################### 27
  13:00  ############## 15
  14:00  ######## 9
  15:00  ################ 17
  16:00  ######### 10
  17:00  ########## 11
  18:00  ############################ 30
  19:00  #################################### 38
  20:00  ######################################## 42
  21:00  ######################### 27
  22:00  ###################### 24
  23:00  ######### 10

20.6% of Food Delivery orders happen 21:00-02:00.
35.8% of Cafe spend happens 08:00-11:00.


## Feature 7 — Anomaly Detection

This flags transactions that are way bigger than usual — but only compared to their own category, using a z-score. A ₹2,000 Swiggy order is definitely unusual, but ₹2,000 on Amazon is nothing. So I had to calculate the average and standard deviation separately for each category instead of lumping everything together.

In [8]:
category_mean = debit_spend.groupby('category')['amount'].transform('mean')
category_std = debit_spend.groupby('category')['amount'].transform('std')
debit_spend['z_score'] = (debit_spend['amount'] - category_mean) / category_std

anomalies = debit_spend[debit_spend['z_score'] > 2].sort_values('z_score', ascending=False)

print(f"Flagged {len(anomalies)} anomalies (z-score > 2).")
print("\nTOP ANOMALIES")
for _, row in anomalies.head(5).iterrows():
    print(f"  {row['date'].strftime('%d %b'):<8} {row['vendor_clean']:<14} Rs. {row['amount']:>8,.0f}  (z={row['z_score']:.1f})  [{row['category']}]")

Flagged 31 anomalies (z-score > 2).

TOP ANOMALIES
  26 Feb   Uncategorised  Rs.    8,383  (z=6.2)  [Uncategorised]
  22 Jun   Uncategorised  Rs.    7,935  (z=5.9)  [Uncategorised]
  14 Jan   Uncategorised  Rs.    7,707  (z=5.7)  [Uncategorised]
  07 Mar   Uncategorised  Rs.    7,366  (z=5.4)  [Uncategorised]
  07 Jan   Uncategorised  Rs.    7,314  (z=5.4)  [Uncategorised]


## Feature 8 — Spending Archetype Detection

This is the fun part — running the numbers through 9 rules to see which "personality" labels actually fit, like whether I'm a Foodie or an Investor. It's not just a vibe check, every single label is backed by an actual number I calculated earlier.

In [9]:
def check_foodie(cat_pct):
    val = cat_pct.reindex(['Food Delivery', 'Restaurants', 'Cafe']).fillna(0).sum()
    return val > 25, val

def check_quick_commerce_junkie(cat_pct):
    val = cat_pct.get('Quick Commerce', 0)
    return val > 15, val

def check_shopaholic(cat_pct):
    val = cat_pct.get('E-commerce', 0)
    return val > 15, val

def check_investor(cat_pct):
    val = cat_pct.get('Investments', 0)
    return val > 15, val

def check_late_night_snacker(late_night_pct):
    return late_night_pct > 50, late_night_pct

def check_cab_commuter(cat_pct):
    val = cat_pct.get('Transport', 0)
    return val > 10, val

def check_subscription_lover(debit_spend_df):
    subs = debit_spend_df[debit_spend_df['category'] == 'Subscriptions']
    n = subs['vendor_clean'].nunique()
    return n >= 5, n

def check_yolo_spender(savings_rate):
    return savings_rate < 10, savings_rate

def check_disciplined_saver(savings_rate):
    return savings_rate > 40, savings_rate

archetype_checks = [
    ("THE FOODIE", check_foodie(cat_pct)),
    ("THE QUICK COMMERCE JUNKIE", check_quick_commerce_junkie(cat_pct)),
    ("THE SHOPAHOLIC", check_shopaholic(cat_pct)),
    ("THE INVESTOR", check_investor(cat_pct)),
    ("THE LATE-NIGHT SNACKER", check_late_night_snacker(late_night_pct)),
    ("THE CAB COMMUTER", check_cab_commuter(cat_pct)),
    ("THE SUBSCRIPTION LOVER", check_subscription_lover(debit_spend)),
    ("THE YOLO SPENDER", check_yolo_spender(savings_rate)),
    ("THE DISCIPLINED SAVER", check_disciplined_saver(savings_rate)),
]

matched_archetypes = []
print("ARCHETYPE RESULTS")
for name, (matched, metric) in archetype_checks:
    flag = "-> MATCH" if matched else "   no match"
    print(f"  {name:<28} metric={metric:6.1f}   {flag}")
    if matched:
        matched_archetypes.append((name, metric))

ARCHETYPE RESULTS
  THE FOODIE                   metric=  14.7      no match
  THE QUICK COMMERCE JUNKIE    metric=   4.0      no match
  THE SHOPAHOLIC               metric=  37.4   -> MATCH
  THE INVESTOR                 metric=  16.4   -> MATCH
  THE LATE-NIGHT SNACKER       metric=  20.6      no match
  THE CAB COMMUTER             metric=   3.0      no match
  THE SUBSCRIPTION LOVER       metric=   4.0      no match
  THE YOLO SPENDER             metric=-229.3   -> MATCH
  THE DISCIPLINED SAVER        metric=-229.3      no match


## Bonus 1 — Vendor Cleanup Audit (+3)

Quick check to see how many transactions I couldn't match to any vendor at all. If it's under 5, I did a good job with my keyword dictionary. If it's more than that, I missed some patterns.

In [10]:
uncategorised = df_clean[df_clean['vendor_clean'] == 'Uncategorised']
print(f"Uncategorised: {len(uncategorised)}")
if len(uncategorised) > 0:
    print(uncategorised['description'].unique())
else:
    print("None — every description mapped to a vendor.")

Uncategorised: 262
['BHIM-BMTC' 'UPI-AMAN-8934@OKAXIS' 'OLA ELECTRIC' 'BMS MOVIE TICKETS'
 'POS OLA-PRIME' 'UPI-THIRDWAVE@OKAXIS' 'BMTC BUS PASS'
 'BANGALORE ELEC SUPPLY' 'TWC INDIA' 'UPI-AMAN-0816@OKAXIS'
 'UPI-CCD@HDFCBANK' 'INSTAMART BANGALORE' 'UPI-VIKAS-6060@OKAXIS'
 'POS BANGALORE RESTAURANT' 'POS CAFE COFFEE DAY' 'POS NYKAA BANGALORE'
 'VI POSTPAID' 'POS BPCL PETROL PUMP' 'TUMMOC-BMTC' 'POS DINEOUT'
 'COFFEE DAY GLOBAL' 'UPI-NYKAA@AXIS' 'UPI-IOC9075@HDFCBANK'
 'FKART INTRNET' 'UPI-RESTAURANT-7285@PAYTM' 'FSN E-COMMERCE'
 'UPI-RESTAURANT-7678@PAYTM' 'INNOVATIVE RETAIL' 'POS INSTAMART'
 'UPI-RESTAURANT-3715@PAYTM' 'STAR INDIA PVT LTD'
 'UPI-RESTAURANT-0251@PAYTM' 'KIRANAKART TECH' 'UPI-RESTAURANT-6377@PAYTM'
 'UPI-VIKAS-5416@OKAXIS' 'UPI-IOC0494@HDFCBANK'
 'UPI-RESTAURANT-4630@PAYTM' 'UPI-IOC1246@HDFCBANK'
 'UPI-RESTAURANT-7888@PAYTM' 'UPI-IOC2000@HDFCBANK'
 'UPI-VI-RECHARGE@HDFCBANK' 'UPI-RESTAURANT-0753@PAYTM' 'NYKAA E-RETAIL'
 'UPI-IOC2115@HDFCBANK' 'UPI-RESTAURANT-3549@PAYTM'


## Bonus 2 — Invented Archetype: THE BREWERY REGULAR (+2)
Rule: 40%+ of Restaurant visits on Fri/Sat evening (19:00-23:00).

In [11]:
restaurants = debit_spend[debit_spend['category'] == 'Restaurants'].copy()
restaurants['day_of_week'] = restaurants['date'].dt.day_name()
weekend_evening_mask = restaurants['day_of_week'].isin(['Friday', 'Saturday']) & restaurants['hour'].between(19, 23)
brewery_pct = weekend_evening_mask.mean() * 100 if len(restaurants) else 0

is_brewery_regular = brewery_pct >= 40
print(f"Weekend-evening restaurant share: {brewery_pct:.1f}%")
print(f"THE BREWERY REGULAR: {'-> MATCH' if is_brewery_regular else 'no match'}")

if is_brewery_regular:
    matched_archetypes.append(("THE BREWERY REGULAR", brewery_pct))

Weekend-evening restaurant share: 14.7%
THE BREWERY REGULAR: no match


## Bonus 3 — Day-of-Week Analysis (+3)

Once I already had time-of-day figured out, it felt natural to also check if weekends cost more per day than weekdays.

In [12]:
debit_spend['day_of_week'] = debit_spend['date'].dt.day_name()
is_weekend = debit_spend['day_of_week'].isin(['Saturday', 'Sunday'])

weekend_total = debit_spend.loc[is_weekend, 'amount'].sum()
weekday_total = debit_spend.loc[~is_weekend, 'amount'].sum()
n_weeks = debit_spend['date'].dt.isocalendar().week.nunique()
weekend_avg_per_day = weekend_total / (n_weeks * 2)
weekday_avg_per_day = weekday_total / (n_weeks * 5)
uplift_pct = (weekend_avg_per_day - weekday_avg_per_day) / weekday_avg_per_day * 100

print(f"Avg weekday spend/day: Rs. {weekday_avg_per_day:,.0f}")
print(f"Avg weekend spend/day: Rs. {weekend_avg_per_day:,.0f}")
print(f"Weekends are {uplift_pct:+.1f}% vs weekdays.")
print(debit_spend.groupby('day_of_week')['amount'].sum().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']).round(0))

Avg weekday spend/day: Rs. 8,324
Avg weekend spend/day: Rs. 8,224
Weekends are -1.2% vs weekdays.
day_of_week
Monday       208950.0
Tuesday      246467.0
Wednesday    294300.0
Thursday     185390.0
Friday       146969.0
Saturday     248715.0
Sunday       178913.0
Name: amount, dtype: float64


## Final Report

This just pulls everything I calculated above into one clean printed report — the kind of thing that actually looks good enough to screenshot and post, instead of just sitting scattered across notebook cells.

In [13]:
BAR_SCALE = 40 / cat_pct.max()

print("=" * 64)
print(" SpendDNA REPORT - RAHUL SHARMA")
print(f" 6 months - {len(df_clean)} transactions - Jan to Jun 2024")
print("=" * 64)

print("\n EXECUTIVE SUMMARY")
print(f"  Total credits : Rs. {total_credit:,.0f}")
print(f"  Total debits  : Rs. {total_debit:,.0f}")
status = "overspending" if net_change < 0 else "saving"
print(f"  Net change    : Rs. {net_change:,.0f} ({status})")
verdict = "BURNING SAVINGS" if savings_rate < 0 else "HEALTHY"
print(f"  Savings rate  : {savings_rate:.1f}% ({verdict})")
print(f"  Transactions  : {len(df_clean)}")
print(f"  Unique vendors: {df_clean['vendor_clean'].nunique()}")

print("\n TOP CATEGORIES")
for cat, amt in cat_totals.head(5).items():
    bar = '#' * int(cat_pct[cat] * BAR_SCALE)
    print(f"  {cat:<16}{bar:<42}{cat_pct[cat]:5.1f}%  Rs. {amt:>9,.0f}")

print("\n TOP VENDORS")
for v, amt in vendor_totals.head(5).items():
    print(f"  {v:<20}Rs. {amt:>10,.0f}  ({vendor_counts[v]:>3} txns)")

print("\n TIME-OF-DAY")
print(f"  Food Delivery late-night: {late_night_pct:.1f}%")
print(f"  Cafe morning runs       : {cafe_morning_pct:.1f}%")

print(f"\n MONTHLY TREND ({month_pivot.columns[0]}->{month_pivot.columns[-1]})")
print(f"  Up fastest  : {biggest_growth_cat} ({growth_pct[biggest_growth_cat]:+.1f}%)")
print(f"  Down fastest: {biggest_decline_cat} ({growth_pct[biggest_decline_cat]:+.1f}%)")

print("\n TOP ANOMALIES")
for _, row in anomalies.head(5).iterrows():
    print(f"  {row['date'].strftime('%d %b'):<8} {row['vendor_clean']:<14} Rs. {row['amount']:>8,.0f} (z={row['z_score']:.1f})")

print("\n ARCHETYPES")
for name, metric in matched_archetypes:
    print(f"  -> {name} (metric: {metric:.1f})")
print("=" * 64)

 SpendDNA REPORT - RAHUL SHARMA
 6 months - 1310 transactions - Jan to Jun 2024

 EXECUTIVE SUMMARY
  Total credits : Rs. 509,774
  Total debits  : Rs. 1,678,901
  Net change    : Rs. -1,169,127 (overspending)
  Savings rate  : -229.3% (BURNING SAVINGS)
  Transactions  : 1310
  Unique vendors: 34

 TOP CATEGORIES
  E-commerce      ########################################   37.4%  Rs.   564,577
  Investments     #################                          16.4%  Rs.   248,160
  Uncategorised   ##############                             13.9%  Rs.   210,491
  Food Delivery   ##########                                 10.0%  Rs.   150,839
  Quick Commerce  ####                                        4.0%  Rs.    60,152

 TOP VENDORS
  Amazon              Rs.    324,303  ( 82 txns)
  Uncategorised       Rs.    210,491  (262 txns)
  Zerodha             Rs.    210,000  ( 14 txns)
  Flipkart            Rs.    170,745  ( 43 txns)
  Swiggy              Rs.     95,523  (223 txns)

 TIME-OF-DAY
  

## Keys Insights & Reflection

This is me stepping back and actually saying what the data means, plus being honest about what was hard (the vendor extractor, definitely) and what I learned along the way — mainly that z-scores only make sense when you calculate them within a category, not across everything at once.